In [ ]:
# Testing Cell
from pathlib import Path

from aviary.api import get_path
from aviary.core.aviary_problem import AviaryProblem
from aviary.interface.cmd_entry_points import _command_map
from aviary.interface.run_aviary import run_aviary
from aviary.utils.doctape import glue_class_methods, glue_keys, glue_variable

if 'run_mission' in _command_map:
    glue_variable('run_mission', md_code=True)

glue_variable(run_aviary.__name__, md_code=True)
glue_variable(f'{run_aviary.__name__}()', md_code=True)
glue_variable(AviaryProblem.__name__, md_code=True)
glue_variable(f'{AviaryProblem.build_model.__name__}()', md_code=True)
glue_variable(f'{AviaryProblem.setup.__name__}()', md_code=True)
# glue_class_methods(AviaryProblem, [], prefix='prob', md_code=True)

Path.exists(get_path('advanced_single_aisle.csv'))
Path.exists(get_path('models/missions/energy_state_default.py'))

# Running a Custom Optimization
Aviary's Python API allows for significantly more customization over what Aviary does. In this example, we will dig further into the Python API to optimize an aircraft with new design variables and constraints.

## The Optimization Problem
In this toy problem, we have two constraints that we want our aircraft to meet. These are arbitrarily chosen for this example, but could represent some specific performance criteria we are trying to meet.
- Thrust-to-Weight ratio greater than 0.35
- Wing loading greater than 120 lbf./ft.<sup>2</sup>

In order for our aircraft to meet these performance goals, we will need to adjust some related properties of the aircraft. The chosen design variables for this problem are:
- Engine scale factor, which will adjust how much thrust our engines are sized to output
- Wing area

We will be using Aviary's default objective function, which minimizes fuel burn over the design mission.

## Setting Up Aviary
For this example we cannot use the {glue:md}`run_mission` command or the {glue:md}`run_aviary()` function in Python. The tradeoff of the "single-line" simplicity of those methods is a lack of customizability over how the problem is set up, which is needed for this problem definition. Instead we will go one level deeper into Aviary's Python API and directly utilize the {glue:md}`AviaryProblem` object<!-- TODO: link to source docs of this object once complete -->. In brief, an {glue:md}`AviaryProblem` is an OpenMDAO Problem that is extended to have additional Aviary-specific features. Creating and running an {glue:md}`AviaryProblem` is very similar to the steps needed to use a pure OpenMDAO problem.

In [ ]:
import aviary.api as av
from aviary.models.missions.energy_state_default import phase_info

# Suppress outputs by setting verbosity as zero (quiet mode)
prob = av.AviaryProblem(verbosity=0)

# Load aircraft and options data from provided sources
prob.load_inputs('validation_cases/validation_data/test_models/aircraft_for_bench_FwFm.csv', phase_info)

# Sanity check inputs and guess initial conditions for mission phases
prob.check_and_preprocess_inputs()

# Have Aviary build the OpenMDAO model with pre-mission, mission, and post-mission components
prob.build_model()

# Selecting optimizer and iteration limit are optional
prob.add_driver('SLSQP', max_iter=20)

# Add the default design variables needed to size the aircraft
prob.add_design_variables()

# Add wing area and engine scaling as additional design variables
prob.model.add_design_var(av.Aircraft.Engine.SCALE_FACTOR, lower=0.8, upper=1.2, ref=1)
prob.model.add_design_var(av.Aircraft.Wing.AREA, lower=1200, upper=1800, units='ft**2', ref=1400)

# Add the default objective function (minimum fuel burn)
prob.add_objective()

# Add constraints for wing loading and thrust-to-weight ratio
prob.model.add_constraint(av.Aircraft.Design.WING_LOADING, lower=120, units='lbf/ft**2')
prob.model.add_constraint(av.Aircraft.Design.THRUST_TO_WEIGHT_RATIO, lower=0.35)

# Standard OpenMDAO problem setup step
prob.setup()

# Run the optimization problem
prob.run_aviary_problem()

In [ ]:
if not prob.result.success:
    raise ValueError('Custom optimization example failed')

This code mirrors what {glue:md}`run_aviary()` is actually doing behind the scenes - these steps can be broken down even further, but that isn't necessary for this example.

There is a combination of {glue:md}`AviaryProblem`-specific methods and some standard OpenMDAO ones mixed together here. `add_design_var()` and `add_constraint()` are two OpenMDAO methods used to define our optimization problem. Because the {glue:md}`AviaryProblem` is based on the OpenMDAO Problem, we can directly use any method that exists for the OpenMDAO Problem too.

The order these steps are executed in is very important! Flipping around the order will break Aviary and result in an immediate error or important parts of the problem definition not being done correctly. The exception is our custom optimization problem setup steps (adding design variables and constraints). In this example, the order does not matter too much, as long as it is done between {glue:md}`build_model()` and {glue:md}`setup()`. In general, when customizing an {glue:md}`AviaryProblem` it is good practice to do any modifications in the same place you would make similar changes to a pure OpenMDAO problem. This takes some familiarity with OpenMDAO to master, but the reward is extremely broad power over exactly what optimization problem Aviary runs.

Now let's take a look at some results of the problem:

In [ ]:
print(f'\nTakeoff Gross Weight = {prob.get_val(av.Mission.GROSS_MASS, units="lbm")[0]} lbm')
print('\nDesign Variables\n---------------')
print(f'Engine Scale Factor (started at 1) = {prob.get_val(av.Aircraft.Engine.SCALE_FACTOR)[0]}')
print(f'Wing Area (started at 1370) = {prob.get_val(av.Aircraft.Wing.AREA, units="ft**2")[0]} ft^2')
print('\nConstraints\n-----------')
print(
    f'Wing Loading = {prob.get_val(av.Aircraft.Design.WING_LOADING, units="lbf/ft**2")[0]} lbf/ft^2'
)
print(f'Thrust/Weight Ratio = {prob.get_val(av.Aircraft.Design.THRUST_TO_WEIGHT_RATIO)[0]}')

We can see that the value for engine scale factor and wing area have changed from their initial values, which is expected since they were design variables for the problem. In addition, the values for our new constraints are right against their specified bounds. This example was designed such that both constraints are simultaneously active, allowing us to easily see how they affected the aircraft design.